# Expyrementor

Expyrementor implements a more modular way to handle design of experiments. The basic usage is designed to be as close to ProcessOptimizer's Optimizer as possible:

In [1]:
from Expyrementor import Expyrementor

space = [[1.0, 100.0], [100, 200], ["cat", "dog", "fish"]]

director = Expyrementor(space)
print(director)
# Asking for the first parameter set to test
first_suggested_params = director.ask()
print(first_suggested_params)
# Telling the director how well the first parameter set performed
director.tell(first_suggested_params[0], 0.5)
# Asking for the next parameter set to test
second_suggested_params = director.ask()
print(second_suggested_params)
# Telling the director how well the second parameter set performed
director.tell(first_suggested_params[0], 0.5)
# Asking for the next two parameter sets to test
third_and_fourth_suggested_params = director.ask(n=2)
print(third_and_fourth_suggested_params)

Expyrementor with a InitialPointStrategizer suggestor.
[[90.10000000000001 170 'dog']]
[[50.5 110 'fish']]
[[70.3 130 'cat']
 [30.7 150 'cat']]


However, we can also make more complicated suggestors. As an example, we are going to make
a suggestor that starts with a 3 point Latin Hypercube Sampling, and after that
randomly chooses a exploring (xi = 10) 20% of the time, ans an exploiting (chi = 0.00001)
Optimizer 80% of the time.

In [2]:
suggestor_definition = {
    "name": "InitialPoint",
    "n_initial_points": 10,
    "initial_suggestor": {
        "name": "LHS", "n_points": 10
    },
    "ultimate_suggestor": {
        "name": "Random",
        "suggestors": [
            {"usage_ratio": 20, "name": "PO", "acq_func_kwargs": {"xi": 10}},
            {"usage_ratio": 80, "name": "PO", "acq_func_kwargs": {"xi": 0.00001}},
        ]
    }
}
director = Expyrementor(space, suggestor_definition)
intial_suggestions = director.ask(7)
director.tell(intial_suggestions, [0.5, 0.6, -0.7, 0.8, 0.9, 1.0, 1.1])
print(director.ask(10))

[[95.05 135 'dog']
 [15.85 105 'dog']
 [85.14999999999999 115 'cat']
 ['33.99800434958006' '104' 'dog']
 ['3.429739342486463' '120' 'fish']
 ['83.55809954099502' '105' 'cat']
 ['12.47231893940161' '170' 'dog']
 ['49.818785128244784' '114' 'cat']
 ['91.57420742670749' '149' 'dog']
 ['21.859534469711146' '136' 'cat']]


You can also make your own suggestor and mix it with the built-in ones. Just make sure
that your suggestor implements the Suggestor protocol.

Specifically, it has to have an `__init__` method, and a `suggest` method which accepts
the input arugments Xi (list of all tested parameters), Yi (list of the results of
testing the parameters) and `n_asked` (the number of suggestions to make).

In [3]:
from Expyrementor.suggestors import Suggestor
from ProcessOptimizer.space import Space, space_factory

class ConstantSuggestor(Suggestor):
    def __init__(self, space: Space, constant: int = 0.5):
        self.space = space
        self.constant = constant

    def suggest(self, Xi, Yi, n_asked):
        return self.space.sample([[self.constant] * len(self.space)]*n_asked)

print(f"ConstantSuggestor is a Suggestor: {issubclass(ConstantSuggestor, Suggestor)}")

space = space_factory([[1.0, 100.0], [100, 200], ["cat", "dog", "fish"]])

suggestor_definition = {
    "name": "InitialPoint",
    "n_initial_points": 5,
    "initial_suggestor": ConstantSuggestor(space),
    "ultimate_suggestor": ConstantSuggestor(space, constant=0.)
    }
director = Expyrementor(space, suggestor_definition)
print(director.ask(10))

ConstantSuggestor is a Suggestor: True
[[50.5 150 'dog']
 [50.5 150 'dog']
 [50.5 150 'dog']
 [50.5 150 'dog']
 [50.5 150 'dog']
 [1.0 100 'cat']
 [1.0 100 'cat']
 [1.0 100 'cat']
 [1.0 100 'cat']
 [1.0 100 'cat']]


Initial Points Suggestor has the option of uding default suggestors for either the
suggestor used to suggest the initial points (initiali suggestor), or the suggestor used
after the intial points are used up (ultimate suggestor), or both.

The default initial suggestor is a Lating Hypercube Sampling suggestor with as many points
as the Initial Point Suggestor will use it.

The default ultimate suggestor is a ProcessOptimizer Optimizer suggestor with no initial
points (the Initial Point Suggestor takes care of the initial points).



In [4]:
space = space_factory([[1.0, 100.0], [100, 200], ["cat", "dog", "fish"]])
suggestor_definition = {
    "name": "InitialPoint",
    "n_initial_points": 11,
    "initial_suggestor": {"name": "Default"},
    "ultimate_suggestor": {"name": "Default"},
    }
director = Expyrementor(space, suggestor_definition)
print(f"Director is a {director}")
print(f"Its suggestor is a {director.suggestor}")
print(f"In turn, that has a {director.suggestor.initial_suggestor} as intial suggestor and a {director.suggestor.ultimate_suggestor} as ultimate suggestor")

Director is a Expyrementor with a InitialPointStrategizer suggestor.
Its suggestor is a InitialPointStrategizer with a LHSSuggestor as inital suggestor and a POSuggestor as ultimate suggestor.
In turn, that has a Latin Hypercube Suggestor with 11 points as intial suggestor and a ProcessOptimizer Suggestor as ultimate suggestor
